# Heart Disease Prediction: End-to-End Clinical Machine Learning Pipeline

**Dataset**: UCI Cleveland Heart Disease Cohort (303 patient records, 14 clinical features)  
**Objective**: Build a leak-free Machine Learning system to predict coronary heart disease, benchmarking regularized linear models, tree bagging, and gradient boosted ensembles with clinical sensitivity / recall prioritization and SHAP explainability.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib

# Ensure project src modules are discoverable
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import COLUMN_NAMES, NUMERIC_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE, RAW_DATA_PATH
from src.data_cleaning import clean_data, report_missing_values, detect_iqr_outliers
from src.preprocessing import get_preprocessor, get_train_test_split

plt.style.use("seaborn-v0_8-whitegrid")
print("Environment and dependencies initialized successfully.")

## 1. Raw Data Ingestion & Clinical Attributes
Load the raw Cleveland dataset directly from `data/raw/` and inspect initial shapes and missing value encodings (`?`).


In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH, header=None, names=COLUMN_NAMES, dtype=str)
print(f"Raw Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

## 2. Quality Assurance & Leak-Free Cleaning
- Convert `?` missing indicators to `np.nan` (known missingness: 4 in `ca`, 2 in `thal`).
- Binarize target: 0 (No Disease, 54.1%) vs 1 (Coronary Disease, 45.9%).
- Check for duplicate rows (0 found).
- Audit outliers via IQR method (outliers retained to preserve clinical signal).
- **Leak-Free Principle**: Zero imputation is performed during cleaning; all imputation is deferred to the pipeline.


In [ ]:
df_clean = clean_data(save_clean=False)
print("Cleaned Dataset Shape:", df_clean.shape)
print("\nTarget Value Counts:")
print(df_clean["target"].value_counts(normalize=True))

## 3. Exploratory Data Analysis (EDA)
Explore key diagnostic relationships:
1. Target class balance
2. Peak exercise heart rate (`thalach`) vs patient age stratified by diagnosis
3. Chest pain paradox (`cp=4` asymptomatic showing high heart disease prevalence)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Target distribution
sns.countplot(data=df_clean, x="target", palette=["#1f77b4", "#d62728"], ax=axes[0], edgecolor="black")
axes[0].set_title("Target Distribution (0: Healthy, 1: Heart Disease)", fontweight="bold")
axes[0].set_xticklabels(["Healthy (54.1%)", "Disease (45.9%)"])
axes[0].set_ylabel("Patient Count")

# Plot 2: Peak Heart Rate vs Age
sns.scatterplot(
    data=df_clean, x="age", y="thalach", hue="target",
    palette=["#1f77b4", "#d62728"], alpha=0.85, ax=axes[1], s=60
)
axes[1].set_title("Peak Heart Rate (thalach) vs Age Stratified by Diagnosis", fontweight="bold")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Peak Exercise Heart Rate (bpm)")
axes[1].legend(title="Diagnosis", labels=["Healthy (0)", "Disease (1)"])

plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline & Stratified Train/Test Split
Construct a leak-free `sklearn.compose.ColumnTransformer`:
- Numeric Pipeline: `SimpleImputer(strategy="median")` -> `StandardScaler()`
- Categorical Pipeline: `SimpleImputer(strategy="most_frequent")` -> `OneHotEncoder(handle_unknown="ignore")`
- Stratified 80/20 train/test split preserving class balance ($45.9\%$ disease in both partitions).


In [ ]:
X_train, X_test, y_train, y_test = get_train_test_split(df=df_clean, save_splits=False)
preprocessor = get_preprocessor()

# Fit strictly on X_train to prevent test leakage
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print(f"X_train shape: {X_train_proc.shape} | Zero NaNs: {np.isnan(X_train_proc).sum() == 0}")
print(f"X_test shape:  {X_test_proc.shape} | Zero NaNs: {np.isnan(X_test_proc).sum() == 0}")
print(f"Total processed feature dimensions: {len(feature_names)}")

## 5. Model Benchmarking & Held-Out Test Evaluation
Load the winning model pipeline (`models/best_model.joblib`) and examine performance on the untouched test partition ($N=61$).


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

best_model = joblib.load(project_root / "models" / "best_model.joblib")
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

print("Classification Report on Held-Out Test Set (N=61, Threshold=0.50):")
print(classification_report(y_test, y_pred, target_names=["Healthy (0)", "Disease (1)"]))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"True Positives:  {cm[1, 1]}  | False Negatives: {cm[1, 0]}")
print(f"False Positives: {cm[0, 1]}  | True Negatives:  {cm[0, 0]}")
print(f"Test Set ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

## 6. Threshold Sensitivity Analysis
Evaluate how precision and recall shift across decision thresholds $p \in [0.10, 0.90]$ to optimize for clinical screening safety (Recall $\ge 90\%$).


In [ ]:
from sklearn.metrics import recall_score, precision_score

thresholds = np.linspace(0.10, 0.90, 81)
recalls = [recall_score(y_test, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]
precisions = [precision_score(y_test, (y_prob >= t).astype(int), zero_division=0) for t in thresholds]

plt.figure(figsize=(9, 5))
plt.plot(thresholds, recalls, label="Recall / Sensitivity", color="#d62728", lw=2.4)
plt.plot(thresholds, precisions, label="Precision / PPV", color="#1f77b4", lw=2.4)
plt.axvline(0.50, color="purple", linestyle="--", label="Operating Threshold (0.50)")

plt.title("Precision-Recall Trade-off Across Decision Thresholds (Favors Recall)", fontsize=13, fontweight="bold")
plt.xlabel("Decision Threshold (p)")
plt.ylabel("Score")
plt.xlim(0.08, 0.92)
plt.ylim(-0.02, 1.05)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 7. Model Interpretability with SHAP
Compute exact Tree SHAP feature attributions on test set patients to visualize the primary drivers of cardiac risk.


In [ ]:
import shap

classifier = best_model.named_steps["classifier"]
explainer = shap.TreeExplainer(classifier)
shap_vals = explainer.shap_values(X_test_proc)

if isinstance(shap_vals, list) and len(shap_vals) == 2:
    shap_matrix = shap_vals[1]
elif isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
    shap_matrix = shap_vals[:, :, 1]
else:
    shap_matrix = shap_vals

plt.figure(figsize=(9, 5))
shap.summary_plot(shap_matrix, X_test_proc, feature_names=feature_names, max_display=10, plot_type="bar", show=False)
plt.title("Top 10 Global Features by Mean Absolute SHAP Impact", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()